# Snowflake PERIOD 型の検証Zenn 記事「Snowflakeに期間型PERIODが登場」のハンズオンです。Snowflake Notebooks にインポートしてそのまま実行できます。最後にクリーンアップのセルがあります。実行後は必ず流してください。

## セットアップ

In [ ]:
CREATE DATABASE IF NOT EXISTS PERIOD_DEMO_DB;USE DATABASE PERIOD_DEMO_DB;CREATE SCHEMA IF NOT EXISTS S1;USE SCHEMA S1;

## ステップ1: PERIOD 値を作る

In [ ]:
SELECT  PERIOD_CONSTRUCT('2026-01-01'::DATE, '2026-02-01'::DATE) AS P_DATE,  PERIOD_CONSTRUCT('09:00:00'::TIME, '18:00:00'::TIME)     AS P_TIME,  PERIOD_CONSTRUCT('2026-01-01 00:00:00'::TIMESTAMP_NTZ,                   '2026-01-02 00:00:00'::TIMESTAMP_NTZ)   AS P_TS;

## ステップ2: 列として持つ

In [ ]:
CREATE OR REPLACE TABLE RESERVATION (  ROOM_ID   STRING,  BOOKED_BY STRING,  STAY      PERIOD(TIMESTAMP_NTZ));INSERT INTO RESERVATIONSELECT 'A', 'sato',       PERIOD_CONSTRUCT('2026-09-10 13:00:00'::TIMESTAMP_NTZ,                        '2026-09-10 15:00:00'::TIMESTAMP_NTZ);INSERT INTO RESERVATIONSELECT 'A', 'suzuki',       PERIOD_CONSTRUCT('2026-09-10 15:00:00'::TIMESTAMP_NTZ,                        '2026-09-10 17:00:00'::TIMESTAMP_NTZ);

In [ ]:
SELECT ROOM_ID, BOOKED_BY, STAY,       PERIOD_BEGIN(STAY) AS B, PERIOD_END(STAY) AS EFROM RESERVATION ORDER BY BOOKED_BY;

## ステップ3: 二重予約を判定する14時から16時で予約を入れたい、という条件です。

In [ ]:
SELECT BOOKED_BYFROM RESERVATIONWHERE ROOM_ID = 'A'  AND PERIOD_OVERLAPS(STAY,        PERIOD_CONSTRUCT('2026-09-10 14:00:00'::TIMESTAMP_NTZ,                         '2026-09-10 16:00:00'::TIMESTAMP_NTZ));

## ステップ4: 境界の扱いを確認する17時から19時で予約を入れたい、という条件です。終了境界を含まないため、結果は0件になります。

In [ ]:
SELECT BOOKED_BYFROM RESERVATIONWHERE ROOM_ID = 'A'  AND PERIOD_OVERLAPS(STAY,        PERIOD_CONSTRUCT('2026-09-10 17:00:00'::TIMESTAMP_NTZ,                         '2026-09-10 19:00:00'::TIMESTAMP_NTZ));

## ステップ5: 重なった時間を取り出す

In [ ]:
SELECT BOOKED_BY,       PERIOD_INTERSECT(STAY,         PERIOD_CONSTRUCT('2026-09-10 14:30:00'::TIMESTAMP_NTZ,                          '2026-09-10 16:30:00'::TIMESTAMP_NTZ)) AS OVERLAPFROM RESERVATION ORDER BY BOOKED_BY;

## 注意ポイント次の2つのセルは**意図的にエラーになります**。記事の「実際に動かして気づいたこと」に対応します。

In [ ]:
-- 002016 (22000): Function MIN does not support PERIOD(TIMESTAMP_NTZ(9)) argument typeSELECT MIN(STAY) FROM RESERVATION;

In [ ]:
-- 003300 (42601): Unsupported type 'PERIOD(TIMESTAMP_NTZ(9))' for clustering keysALTER TABLE RESERVATION CLUSTER BY (STAY);

## クリーンアップ検証で作ったオブジェクトを削除します。必ず実行してください。

In [ ]:
DROP DATABASE IF EXISTS PERIOD_DEMO_DB;